In [12]:
import torch
import torchvision.models as models
import backbone.Test as test
import timm
import backbone.Custom as Custom
from torch.utils.data import Dataset, DataLoader
import backbone.VISUAL as viz
import backbone.GalaxyZoo as gz
import importlib
import backbone.AstroMLmodified as AstroMLmod
importlib.reload(gz)


<module 'backbone.GalaxyZoo' from '/users/koketso/Feature_extraction/spectra_for_features/backbone/GalaxyZoo.py'>

# Data

In [13]:
train,val,class_  = gz.Galaxy_zoo_data_loaders()

In [ ]:
## Model: Zoobot ConvNext Base

encoder = timm.create_model('hf_hub:mwalmsley/zoobot-encoder-convnext_base', pretrained=True, num_classes=0)


zoobot_rep, zoobot_label, zoobot_ids = Custom.get_representations(model  = encoder,
                           loader = val,
                           encoder = True,
                           labeled = False)

zoobot_rep_c, zoobot_label_c, zoobot_ids_c = Custom.get_representations(model  = encoder,
                           loader = class_,
                           encoder = True,
                           labeled = True)

umap_  =viz.umap(zoobot_rep)
c_accuracy = test.clustering_accuracy(zoobot_rep,zoobot_ids),(zoobot_label_c,zoobot_ids_c))
zoobot_tpcf = AstroMLmod.TPCF_score(zoobot_rep)
zoobot_id = AstroMLmod.id_score(zoobot_rep)

## Model: Dino ConvNext base

In [ ]:
dinov3_convnext_base = torch.hub.load("../dinov3", 'dinov3_convnext_base',
                                      source='local',
                                      weights="dinov3_convnext_base_pretrain_lvd1689m-801f2ba9.pth")
dino_rep, dino_label, dino_ids = Custom.get_representations(model  = dinov3_convnext_base,
                           loader = val,
                           encoder = True,
                           labeled = False)

dino_rep_c, dino_label_c, dino_ids_c = Custom.get_representations(model  = encoder,
                           loader = class_,
                           encoder = True,
                           labeled = True)

umap_  =viz.umap(dino_rep)

c_accuracy = test.clustering_accuracy(dino_rep,zoobot_ids),(dino_label_c,dino_ids_c))
dino_tpcf = AstroMLmod.TPCF_score(dino_rep)
dono_id = AstroMLmod.id_score(dino_rep)

## Clustering accuracy

In [ ]:
## Model: Dino ConvNext base
import numpy as np
from collections import Counter

def clustering_accuracy(y_true, y_pred):
    """
    Compute clustering accuracy by assigning the most frequent true label to each cluster.
    Args:
        y_true: Array of ground truth labels
        y_pred: Array of cluster assignments
    Returns:
        Accuracy score (float between 0 and 1)
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    correct = 0
    
    # For each cluster, find the most frequent true label
    for cluster_id in np.unique(y_pred):
        # Get all true labels within this cluster
        cluster_mask = (y_pred == cluster_id)     
        labels_in_cluster = y_true[cluster_mask]
        
        # Find the most common label in this cluster
        most_common_label = Counter(labels_in_cluster).most_common(1)[0][0]
        
        # Count how many samples match the most common label
        correct += np.sum(labels_in_cluster == most_common_==label)
    
    # Return accuracy as ratio of correct assignments
    return correct / len(y_true)

y_true = [0, 0, 1, 1, 2, 2,2]
y_pred = [1, 1, 0, 0, 2, 2]

accuracy = clustering_accuracy(y_true, y_pred)
print(f"Clustering Accuracy: {accuracy:.2f}")  #

In [1]:

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

IndexError: boolean index did not match indexed array along dimension 0; dimension is 7 but corresponding boolean dimension is 6

In [ ]:
indices = [original_array.index(item) for item in subset_array if item in original_array]